In [17]:
import os, json, torch, librosa, numpy as np, soundfile as sf
from transformers import AutoFeatureExtractor, AutoModelForAudioClassification
from IPython.display import Audio as AudioPlayer, display

device           = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MAX_AUDIO_LENGTH = 16000 * 5
print(f'Device: {device}')

# ── Find model paths automatically ───────────────────────────────────
print('\nSearching for models...')
model_paths = {

    'Audio Model' : '/kaggle/input/datasets/ziadmohamed11/final-wavlmlarge-4-datasets/emotion-recognition-final',





}

for name, path in model_paths.items():
    print(f'  {name} → {path}')

# ── Load both models ──────────────────────────────────────────────────
models = {}
for name, path in model_paths.items():
    print(f'\nLoading {name}...')
    feature_extractor = AutoFeatureExtractor.from_pretrained(path)
    model             = AutoModelForAudioClassification.from_pretrained(path)
    model.eval().to(device)
    with open(f'{path}/label_mapping.json', 'r') as f:
        label_mapping = json.load(f)
    models[name] = {
        'feature_extractor': feature_extractor,
        'model'            : model,
        'label_mapping'    : label_mapping,
    }
    print(f'  ✅ {name} loaded!')

# ── Predict function (works for any model) ────────────────────────────
def predict_with_model(audio_path, model_name):
    m                 = models[model_name]
    feature_extractor = m['feature_extractor']
    model             = m['model']
    label_mapping     = m['label_mapping']

    audio, sr = librosa.load(audio_path, sr=feature_extractor.sampling_rate)
    inputs    = feature_extractor(
        audio,
        sampling_rate=sr,
        return_tensors='pt',
        padding=True,
        max_length=MAX_AUDIO_LENGTH,
        truncation=True
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
    probabilities     = torch.nn.functional.softmax(logits, dim=-1)[0]
    predicted_id      = torch.argmax(probabilities).item()
    predicted_emotion = label_mapping['id2label'][str(predicted_id)]
    confidence        = probabilities[predicted_id].item()
    emotion_scores    = {
        label_mapping['id2label'][str(idx)]: prob.item()
        for idx, prob in enumerate(probabilities)
    }
    return {
        'emotion'   : predicted_emotion,
        'confidence': confidence,
        'all_scores': emotion_scores
    }

def display_prediction(result, model_name):
    print(f'\n{"=" * 60}')
    print(f'EMOTION PREDICTION RESULTS — {model_name.upper()}')
    print('=' * 60)
    print(f"\n  Predicted Emotion : {result['emotion'].upper()}")
    print(f"  Confidence        : {result['confidence']*100:.2f}%")
    print('\n' + '-' * 60)
    print('  All Emotion Scores:')
    print('-' * 60)
    for emotion, score in sorted(result['all_scores'].items(), key=lambda x: -x[1]):
        bar = '█' * int(score * 40) + '░' * (40 - int(score * 40))
        print(f'  {emotion:12s} [{bar}] {score*100:5.2f}%')
    print('=' * 60)

print('\n✅models ready!')

Device: cpu

Searching for models...
  Audio Model → /kaggle/input/datasets/ziadmohamed11/final-wavlmlarge-4-datasets/emotion-recognition-final

Loading Audio Model...


Loading weights:   0%|          | 0/492 [00:00<?, ?it/s]

  ✅ Audio Model loaded!

✅models ready!


In [19]:
import librosa, soundfile as sf
import numpy as np

AUDIO_PATH = '/kaggle/input/datasets/ziadmohamed11/abdodaheh/.mp3'  # ← your path
audio_array, sr = librosa.load(AUDIO_PATH, sr=16000, mono=True)
duration = len(audio_array) / sr
print(f"✅ Audio loaded: {duration:.1f}s at {sr}Hz")

✅ Audio loaded: 54.3s at 16000Hz


In [20]:
import json

# ── Load the output from Script Alignment notebook ────────────────────────────
ALIGNED_PATH = '/kaggle/input/datasets/ziadmohamed11/abdoaligggg/sentences_aligned (4).json'

with open(ALIGNED_PATH, encoding='utf-8') as f:
    sentences_aligned = json.load(f)

print(f"✅ Loaded {len(sentences_aligned)} aligned sentences:")
for i, s in enumerate(sentences_aligned):
    icon = {'ok': '✅', 'partial': '🟡', 'missing': '🔴'}.get(s['status'], '❓')
    t    = f"{s['t_start']:.2f}s → {s['t_end']:.2f}s" if s['t_start'] else 'MISSING'
    print(f"  [{i+1:2}] {icon} {s['emotion'].upper():10s} | {t} | {s['content'][:45]}")

✅ Loaded 4 aligned sentences:
  [ 1] ✅ NEUTRAL    | 0.12s → 7.07s | عبد السلام عاطف من جامعة الزقازيق بيقول احنا 
  [ 2] ✅ HAPPY      | 7.24s → 18.51s | علشان ساعتها هتلاقي اللحمة طعمها بقي أطعم لأن
  [ 3] ✅ HAPPY      | 18.64s → 46.14s | قول له عايز لحمة من البر حتة كده تندر على ذوق
  [ 4] ✅ HAPPY      | 46.21s → 54.23s | بس كده كشة فلما تيجي تخرج اللحمة لازم إن انت 


In [21]:
from collections import Counter
from IPython.display import HTML
import soundfile as sf

emotion_colors = {
    'angry':'#ff4d4d', 'calm':'#4fc3f7', 'disgust':'#81c784',
    'fearful':'#ba68c8', 'happy':'#ffd54f', 'neutral':'#90a4ae',
    'sad':'#5c6bc0', 'surprised':'#ff8a65',
}

def badge(emo, conf=None):
    c   = emotion_colors.get(emo.lower(), '#888')
    sup = f'<br><small>{conf:.0f}%</small>' if conf is not None else ''
    return (f'<span style="background:{c};color:#000;padding:2px 9px;'
            f'border-radius:10px;font-weight:600;font-size:12px">'
            f'{emo.upper()}{sup}</span>')

def conf_bar(val, color='#4fc3f7'):
    return (f'<div style="background:#2a2a2a;border-radius:5px;height:16px;min-width:80px">'
            f'<div style="background:{color};width:{val:.0f}%;height:100%;border-radius:5px;'
            f'text-align:center;font-size:11px;line-height:16px;color:#000;font-weight:600">'
            f'{val:.1f}%</div></div>')

# ── Predict per sentence ───────────────────────────────────────────────────────
model_names = list(models.keys())   # all loaded models
results     = []
min_samples = int(0.3 * sr)        # skip segments shorter than 0.3s

for idx, entry in enumerate(sentences_aligned):

    # ── Skip lines actor never said ───────────────────────────────────────────
    if entry['status'] == 'missing' or entry['t_start'] is None:
        print(f"  [{idx+1:2}] ⏭️  Skipped — actor did not say this line | {entry['content'][:45]}")
        continue

    # ── Slice audio using librosa array (not torchaudio waveform) ─────────────
    start_sample = int(entry['t_start'] * sr)
    end_sample   = int(entry['t_end']   * sr)
    segment      = audio_array[start_sample:end_sample]   # shape: (samples,)

    # ── Skip if segment too short ─────────────────────────────────────────────
    if len(segment) < min_samples:
        print(f"  [{idx+1:2}] ⚠️  Too short ({len(segment)/sr:.2f}s) — skipping | {entry['content'][:45]}")
        continue

    # ── Save temp wav and predict ─────────────────────────────────────────────
    tmp_path = f'/tmp/sent_{idx}.wav'
    sf.write(tmp_path, segment, sr)

    preds = {name: predict_with_model(tmp_path, name) for name in model_names}

    results.append({
        'sentence': entry['content'],
        'expected': entry['emotion'],
        'status'  : entry['status'],
        'coverage': entry.get('coverage', 1.0),
        'time'    : f"{entry['t_start']:.1f}s–{entry['t_end']:.1f}s",
        'preds'   : preds,
    })

    cov_str  = f"{entry.get('coverage', 1.0)*100:.0f}%"
    icon     = '✅' if entry['status'] == 'ok' else '🟡'
    preds_str = ' | '.join(f"{n[:14]}: {preds[n]['emotion'].upper()} ({preds[n]['confidence']*100:.0f}%)"
                            for n in model_names)
    print(f"  [{idx+1:2}] {icon} expected={entry['emotion'].upper():10s} | "
          f"{entry['t_start']:.1f}s–{entry['t_end']:.1f}s | cov={cov_str} | {preds_str}")

print(f"\n✅ Predicted {len(results)} sentences")

# ── HTML results table ────────────────────────────────────────────────────────
mc = {m: ['#4fc3f7','#81c784','#ffd54f','#ff8a65','#ce93d8'][i % 5]
      for i, m in enumerate(model_names)}

html = '''<style>
  .rt{border-collapse:collapse;width:100%;font-family:monospace;font-size:12px}
  .rt th{background:#1a1a2e;color:#fff;padding:8px 12px;border:1px solid #333;text-align:center}
  .rt td{padding:7px 12px;border:1px solid #2a2a2a;background:#111;color:#eee;text-align:center}
  .rt tr:hover td{background:#1a1a2e}
</style>
<table class="rt"><thead><tr>
  <th style="text-align:left">Sentence</th>
  <th>Time</th>
  <th>Coverage</th>
  <th>Expected</th>'''

for m in model_names:
    html += f'<th style="color:{mc[m]}">{m.upper()[:18]}</th>'
html += '</tr></thead><tbody>'

total_correct = {m: 0 for m in model_names}

for row in results:
    # dim out partial rows slightly
    row_style = 'opacity:0.75' if row['status'] == 'partial' else ''
    cov_color = '#ffd54f' if row['status'] == 'partial' else '#4caf50'
    cov_str   = f"{row['coverage']*100:.0f}%"

    html += (f'<tr style="{row_style}">'
             f'<td style="text-align:right;direction:rtl;max-width:300px;white-space:normal;word-wrap:break-word">{row["sentence"]}</td>'
             f'<td style="color:#888;font-size:11px;white-space:nowrap">{row["time"]}</td>'
             f'<td style="color:{cov_color};font-weight:700">{cov_str}</td>'
             f'<td>{badge(row["expected"])}</td>')

    for m in model_names:
        pred   = row['preds'][m]
        hit    = pred['emotion'].lower() == row['expected'].lower()
        if hit: total_correct[m] += 1
        border = '2px solid #4caf50' if hit else '2px solid #ef5350'
        html  += f'<td style="border:{border}">{badge(pred["emotion"], pred["confidence"]*100)}</td>'

    html += '</tr>'

# ── Accuracy row ──────────────────────────────────────────────────────────────
html += '<tr><td colspan="4" style="font-weight:700;color:#ffd54f;background:#111">Accuracy</td>'
for m in model_names:
    acc   = total_correct[m] / len(results) * 100 if results else 0
    html += (f'<td style="font-weight:700;color:#ffd54f;background:#111">'
             f'{acc:.0f}% ({total_correct[m]}/{len(results)})</td>')
html += '</tr></tbody></table>'

display(HTML(html))

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  [ 1] ✅ expected=NEUTRAL    | 0.1s–7.1s | cov=96% | Audio Model: HAPPY (95%)
  [ 2] ✅ expected=HAPPY      | 7.2s–18.5s | cov=91% | Audio Model: HAPPY (100%)
  [ 3] ✅ expected=HAPPY      | 18.6s–46.1s | cov=96% | Audio Model: HAPPY (85%)
  [ 4] ✅ expected=HAPPY      | 46.2s–54.2s | cov=96% | Audio Model: ANGRY (100%)

✅ Predicted 4 sentences


In [22]:
import json
import pandas as pd
from IPython.display import display, HTML, Audio as AudioPlayer

# ── Load comparison df ────────────────────────────────────────────────────────
with open('/kaggle/input/datasets/ziadmohamed11/compppp/comparison_df.json', encoding='utf-8') as f:
    df = pd.DataFrame(json.load(f))

# ── Tag df rows with sentence index ──────────────────────────────────────────
def annotate_sent_idx(df, sentences_aligned):
    sent_word_map = []
    for s_idx, entry in enumerate(sentences_aligned):
        sent_word_map.extend([s_idx] * len(entry['content'].split()))

    cursor, last, indices = 0, 0, []
    for _, row in df.iterrows():
        if row['Script'] != '-':
            if cursor < len(sent_word_map):
                last = sent_word_map[cursor]
                cursor += 1
        indices.append(last)
    df = df.copy()
    df['sent_idx'] = indices
    return df

df = annotate_sent_idx(df, sentences_aligned)

# ── Styling helpers ───────────────────────────────────────────────────────────
STATUS_COLOR = {
    '✓ Match'   : '#2e7d32',
    '🟡 Changed' : '#f57f17',
    '🔴 Skipped' : '#c62828',
    '🟢 Added'   : '#1565c0',
}
emotion_colors = {
    'angry':'#ff4d4d', 'calm':'#4fc3f7', 'disgust':'#81c784',
    'fearful':'#ba68c8', 'happy':'#ffd54f', 'neutral':'#90a4ae',
    'sad':'#5c6bc0', 'surprised':'#ff8a65',
}

def badge(emo):
    c = emotion_colors.get(emo.lower(), '#888')
    return (f'<span style="background:{c};color:#000;padding:2px 9px;'
            f'border-radius:10px;font-weight:600;font-size:12px">{emo.upper()}</span>')

def word_chip(word, status):
    c = STATUS_COLOR.get(status, '#888')
    return (f'<span style="background:{c}22;color:{c};border:1px solid {c};'
            f'padding:2px 7px;border-radius:5px;margin:2px;display:inline-block;'
            f'font-size:13px;direction:rtl">{word}</span>')

# ── Render each sentence ──────────────────────────────────────────────────────
for i, entry in enumerate(sentences_aligned):

    # ── Missing sentence ──────────────────────────────────────────────────────
    if entry['status'] == 'missing':
        display(HTML(
            f'<div style="margin-bottom:18px;padding:12px;background:#111;'
            f'border-radius:8px;border-left:4px solid #c62828">'
            f'<b style="color:#c62828">[{i+1}] MISSING</b> — actor did not say this line<br>'
            f'<span style="color:#888;direction:rtl;display:block;margin-top:6px">'
            f'{entry["content"]}</span></div>'
        ))
        continue

    # ── Word chips ────────────────────────────────────────────────────────────
    sent_df = df[df['sent_idx'] == i]

    script_chips = ''.join(
        word_chip(row['Script'], row['Status'])
        for _, row in sent_df[sent_df['Script'] != '-'].iterrows()
    )
    trans_chips = ''.join(
        word_chip(row['Transcript'], row['Status'])
        for _, row in sent_df[sent_df['Transcript'] != '-'].iterrows()
    )

    # ── Detected emotion ──────────────────────────────────────────────────────
    detected = next((r for r in results if r['sentence'] == entry['content']), None)
    det_html = ''
    if detected:
        for mn in model_names:
            pred = detected['preds'].get(mn, {})
            em   = pred.get('emotion', 'N/A')
            cf   = pred.get('confidence', 0.0)
            hit  = em.lower() == entry['emotion'].lower()
            border = '2px solid #4caf50' if hit else '2px solid #ef5350'
            det_html += (f'<span style="border:{border};padding:3px 8px;border-radius:6px;'
                         f'margin-right:8px;font-size:11px;color:#eee">'
                         f'{mn[:16]}: {badge(em)} {cf*100:.0f}%</span>')

    cov   = entry.get('coverage', 1.0)
    t_str = f"{entry['t_start']:.2f}s → {entry['t_end']:.2f}s"
    cov_c = '#ffd54f' if entry['status'] == 'partial' else '#4caf50'
    ec    = emotion_colors.get(entry['emotion'].lower(), '#444')

    # ── Render card ───────────────────────────────────────────────────────────
    display(HTML(f'''
    <div style="margin-bottom:6px;padding:12px;background:#111;
                border-radius:8px;border-left:4px solid {ec}">
      <div style="display:flex;justify-content:space-between;align-items:center;margin-bottom:8px">
        <span style="color:#aaa;font-size:12px">
          <b style="color:#fff">[{i+1}]</b> &nbsp;
          {t_str} &nbsp;|&nbsp;
          <span style="color:{cov_c}">coverage {cov*100:.0f}%</span>
        </span>
        <span>Expected: {badge(entry['emotion'])} &nbsp; Detected: {det_html}</span>
      </div>
      <div style="margin-bottom:6px">
        <span style="color:#888;font-size:11px">SCRIPT &nbsp;&nbsp;</span>
        <span style="direction:rtl;display:inline-block">{script_chips}</span>
      </div>
      <div>
        <span style="color:#888;font-size:11px">ACTOR &nbsp;&nbsp;&nbsp;</span>
        <span style="direction:rtl;display:inline-block">{trans_chips}</span>
      </div>
    </div>'''))

    # ── Play audio segment ────────────────────────────────────────────────────
    start_sample = int(entry['t_start'] * sr)
    end_sample   = int(entry['t_end']   * sr)
    segment      = audio_array[start_sample:end_sample]
    display(AudioPlayer(segment, rate=sr, autoplay=False))